# Rented-GPU condition of the RMT A3 benchmark (Google Colab Pro)

Team KaiyrbekulyAitbek_TurganbekNurhan, AITU SE-2424. Repository: https://github.com/Newterios/rmt-llm-inference-benchmark

**Before running:** Runtime > Change runtime type > **L4 GPU** (24 GB). Then Runtime > Run all.
Total time about 60 to 70 minutes (build 8 min, model download 3 min, main protocol about 50 min).
Keep the Colab tab open. When finished, the results are copied to the shared Drive folder and offered for download.

Write down the **compute units per hour** shown in the Resources panel (top right, "Change runtime type" area): it is the price of this condition.

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,power.limit --format=csv
import datetime, platform; print(datetime.datetime.now(), platform.platform())

In [ ]:
%cd /content
!rm -rf rmt-llm-inference-benchmark && git clone -q https://github.com/Newterios/rmt-llm-inference-benchmark.git
%cd /content/rmt-llm-inference-benchmark
!pip install -q httpx pandas numpy tqdm psutil python-dotenv anthropic matplotlib
!git log --oneline | head -3

In [ ]:
# Build llama.cpp (llama-server) with CUDA. About 8 minutes on an L4 runtime.
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release -DLLAMA_CURL=OFF > /content/cmake_configure.log 2>&1
!cmake --build /content/llama.cpp/build --target llama-server -j 8 > /content/cmake_build.log 2>&1 || tail -30 /content/cmake_build.log
!ls -la /content/llama.cpp/build/bin/llama-server && /content/llama.cpp/build/bin/llama-server --version 2>&1 | head -2
!cd /content/llama.cpp && git rev-parse --short HEAD && git log -1 --format=%cd

In [ ]:
# Same model file as on the Mac and the PC (sha256 must match 7b064f58...).
!mkdir -p models && curl -L -s -o models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf https://huggingface.co/bartowski/Meta-Llama-3.1-8B-Instruct-GGUF/resolve/main/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf
!ls -la models/ && sha256sum models/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf

In [ ]:
# Start llama-server with the identical configuration (16 slots x 1024 tokens, prompt cache off).
# Port 8090: Colab occupies 8080 itself. Defines `server` and `power` used by the last cell.
!git pull -q
exec(open("bench/cloud/colab_start_server.py").read())

In [ ]:
# Main protocol first (the part the report needs most), then the pilot protocol if time allows.
!python bench/run.py --condition rented_colab_gpu --task throughput --set main --concurrency 1 4 16 --repeats 3 --note "Colab Pro runtime"
!python bench/run.py --condition rented_colab_gpu --task quality    --set main --concurrency 4 --repeats 2 --note "Colab Pro runtime"
!nvidia-smi --query-gpu=name,power.draw,memory.used --format=csv

In [ ]:
# Pilot protocol (optional, about 12 minutes). Skip this cell if the session is close to its limit.
!python bench/run.py --condition rented_colab_gpu --task throughput --set pilot --concurrency 1 4 16 --repeats 2 --note "Colab Pro runtime"
!python bench/run.py --condition rented_colab_gpu --task quality    --set pilot --concurrency 4 --repeats 1 --note "Colab Pro runtime"

In [ ]:
# Stop loggers, pack the results, copy them to the shared Drive folder and offer a download.
import subprocess; subprocess.run("pkill -f nvidia-smi; pkill -f llama-server", shell=True)
!cp runs/rented_server.log runs/rented_colab_gpu/ && cp /content/cmake_build.log runs/rented_colab_gpu/ 2>/dev/null
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv > runs/rented_colab_gpu/gpu_info.txt
!cd runs && zip -q -r /content/rented_colab_gpu_runs.zip rented_colab_gpu && ls -la /content/rented_colab_gpu_runs.zip
from google.colab import drive, files
drive.mount('/content/drive')
import shutil, glob, os
dst = "/content/drive/MyDrive/RMT_A3_KaiyrbekulyAitbek_TurganbekNurhan_data/colab"
os.makedirs(dst, exist_ok=True)
shutil.copy("/content/rented_colab_gpu_runs.zip", dst)
print("copied to", dst, os.listdir(dst))
files.download("/content/rented_colab_gpu_runs.zip")

## What to record for the report
1. The GPU name and the compute units per hour shown by Colab (write it in `bench/prices.json`, key `rented_colab_gpu.compute_units_per_hour`).
2. Screenshot of the cell with `nvidia-smi` and of the final `done:` line of the main run (figures for Segment 1 T7).
3. Share this notebook: File > Share > Anyone with the link, and record the link in the report.